# Cirrhosis Patient Survival Prediction Notebook
### Задачи:
- Предобработка данных
- Обучение baseline RandomForest
- Optuna Hyperparameter Optimization для CatBoost
- Обучение финальной CatBoost модели
- Сохранение предсказаний
- Логирование через ClearML и logging

In [ ]:
# Импорты
import os
import logging
import joblib
import optuna
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
from clearml import Task

os.makedirs('./data', exist_ok=True)
logging.basicConfig(filename='./data/log_file.log', level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# ClearML Task
task = Task.init(project_name='LiverCirrhosis', task_name='Notebook_Training', output_uri=None)
logger = task.get_logger()

In [ ]:
# Загрузка данных
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
train_df.head()

In [ ]:
# Категориальные и численные признаки
cat_features = ['Sex', 'Drug', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
num_features = ['Age', 'N_Days', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage']

In [ ]:
# Предобработка данных
le = LabelEncoder()
y = le.fit_transform(train_df['Status'])
X = train_df.drop(['Status','id'], axis=1)
X[cat_features] = X[cat_features].astype(str).fillna('Missing')
X[num_features] = X[num_features].fillna(X[num_features].median())

In [ ]:
# Baseline RandomForest
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=300, random_state=42)
scores = []
for train_idx, val_idx in kf.split(X, y):
    rf.fit(X.iloc[train_idx], y[train_idx])
    preds = rf.predict_proba(X.iloc[val_idx])
    scores.append(log_loss(y[val_idx], preds))

baseline_logloss = np.mean(scores)
print('Baseline RandomForest CV LogLoss:', baseline_logloss)
logger.report_text(f'Baseline RandomForest CV LogLoss: {baseline_logloss}')

In [ ]:
# Оптимизация CatBoost через Optuna
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'loss_function': 'MultiClass',
        'verbose': 0
    }
    scores = []
    for train_idx, val_idx in kf.split(X, y):
        model = CatBoostClassifier(**params)
        model.fit(X.iloc[train_idx], y[train_idx], cat_features=cat_features)
        scores.append(log_loss(y[val_idx], model.predict_proba(X.iloc[val_idx])))
    return np.mean(scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)
best_params = study.best_params
print('Best CatBoost params:', best_params)
logger.report_text(f'Best CatBoost params: {best_params}')

In [ ]:
# Финальная CatBoost модель
model = CatBoostClassifier(**best_params, loss_function='MultiClass', verbose=0)
model.fit(X, y, cat_features=cat_features)

# CV LogLoss на всех данных
preds = model.predict_proba(X)
final_logloss = log_loss(y, preds)
print('Final CatBoost LogLoss:', final_logloss)
logger.report_text(f'Final CatBoost LogLoss: {final_logloss}')

In [ ]:
# Сохранение модели и предсказаний
joblib.dump(model, './model/catboost_model.pkl')
joblib.dump(le, './model/label_encoder.pkl')

X_test = test_df.drop(['id'], axis=1)
X_test[cat_features] = X_test[cat_features].astype(str).fillna('Missing')
X_test[num_features] = X_test[num_features].fillna(X_test[num_features].median())

preds_test = model.predict_proba(X_test)
preds_df = pd.DataFrame(preds_test, columns=le.classes_)
preds_df.insert(0, 'id', test_df['id'])
preds_df.columns = ['id', 'Status_C', 'Status_CL', 'Status_D']
preds_df.to_csv('./data/results.csv', index=False)
print('Test predictions saved to ./data/results.csv')